In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.9 MB/s eta 0:00:00


In [2]:
import os
import json
import torch
import numpy as np
import faiss
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import torch.nn.functional as F
from collections import defaultdict
import shutil

# ─────────────────────────────────────────
# 0. HuggingFace Authentication
# ─────────────────────────────────────────
try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"]               = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ HuggingFace login successful.")
except Exception as e:
    print(f"⚠️  HF token not found: {e}")
    HF_TOKEN = None

# ─────────────────────────────────────────
# 1. GPU Check
# ─────────────────────────────────────────
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — "
          f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

DEVICE = "cuda:0"

# ─────────────────────────────────────────
# 2. Paths
# ─────────────────────────────────────────
raw_dataset_dir     = '/kaggle/input/datasets/hades199/vr-final-project-dataset/vr-final-project-dataset'
partition_file      = os.path.join(raw_dataset_dir, 'Eval/list_eval_partition.txt')
bbox_file           = os.path.join(raw_dataset_dir, 'Anno/list_bbox_inshop.txt')
cropped_dir         = '/kaggle/input/datasets/hades199/3c-yolo-cropped-images'
faiss_index_dir     = '/kaggle/input/datasets/hades199/3c-faiss-indexes/faiss_indexes'
clip_finetuned_base = '/kaggle/input/datasets/hades199/3c-clip-fintuned-model/clip_finetuned_v2'
results_dir         = '/kaggle/working/eval_results'
CLIP_CACHE          = '/kaggle/working/cache/clip_base'

os.makedirs(results_dir, exist_ok=True)
os.makedirs(CLIP_CACHE,  exist_ok=True)

# ─────────────────────────────────────────
# 3. Config
# ─────────────────────────────────────────
SEEDS        = [42, 601, 606, 619, 623]
ALPHA_VALUES = [0.5, 0.7]
K_VALUES     = [5, 10, 15]
TOP_K_SEARCH = 50   

# ─────────────────────────────────────────
# 4. Download base CLIP once → cache → offline
# ─────────────────────────────────────────
os.environ["TRANSFORMERS_OFFLINE"] = "0"

if not os.path.exists(os.path.join(CLIP_CACHE, 'config.json')):
    print("Downloading base CLIP (one time only)...")
    kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}
    m = CLIPModel.from_pretrained(
        "openai/clip-vit-base-patch32", **kwargs)
    p = CLIPProcessor.from_pretrained(
        "openai/clip-vit-base-patch32", **kwargs)
    m.save_pretrained(CLIP_CACHE)
    p.save_pretrained(CLIP_CACHE)
    del m
    torch.cuda.empty_cache()
    print(f"✅ CLIP cached at {CLIP_CACHE}")
else:
    print(f"✅ CLIP already cached at {CLIP_CACHE}")

os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("🔒 Offline mode ON — no more network calls.\n")

# ─────────────────────────────────────────
# 5. Load Partition & Map Classes
# ─────────────────────────────────────────
print("Loading partition and class metadata...")
df_part    = pd.read_csv(partition_file, sep=r'\s+', skiprows=1)
df_bbox    = pd.read_csv(bbox_file, sep=r'\s+', skiprows=1)

# Merge to get clothes_type (1: Upper, 2: Lower, 3: Full)
df_part = pd.merge(df_part, df_bbox[['image_name', 'clothes_type']], on='image_name', how='left')

class_mapping = {1: 'Upper', 2: 'Lower', 3: 'Full'}
df_part['class_name'] = df_part['clothes_type'].map(class_mapping)

df_query   = df_part[df_part['evaluation_status'] == 'query'].reset_index(drop=True)
df_gallery = df_part[df_part['evaluation_status'] == 'gallery'].reset_index(drop=True)

def extract_item_id(image_name):
    parts = image_name.replace('\\', '/').split('/')
    for part in parts:
        if part.startswith('id_'):
            return part
    return None

df_query['item_id']   = df_query['image_name'].apply(extract_item_id)
df_gallery['item_id'] = df_gallery['image_name'].apply(extract_item_id)
df_query   = df_query.dropna(subset=['item_id']).reset_index(drop=True)
df_gallery = df_gallery.dropna(subset=['item_id']).reset_index(drop=True)

print(f"Query images    : {len(df_query)}")
print(f"Gallery images  : {len(df_gallery)}")

# ─────────────────────────────────────────
# 6. Ground truth relevance counts
# ─────────────────────────────────────────
item_id_to_gallery_count = df_gallery.groupby('item_id').size().to_dict()

# ─────────────────────────────────────────
# 7. Helper
# ─────────────────────────────────────────
def resolve_crop_path(image_name):
    p1 = os.path.join(cropped_dir, image_name)
    p2 = os.path.join(cropped_dir, image_name.replace('/', '_'))
    if os.path.exists(p1): return p1
    if os.path.exists(p2): return p2
    return None

# ─────────────────────────────────────────
# 8. Query Embedding
# ─────────────────────────────────────────
def get_query_embedding(clip_model, processor, cropped_img, device):
    inputs = processor(images=cropped_img, return_tensors="pt").to(device)
    with torch.no_grad():
        vision_outputs = clip_model.vision_model(pixel_values=inputs["pixel_values"])
        image_embeds = clip_model.visual_projection(vision_outputs.pooler_output)
    image_embeds = F.normalize(image_embeds, dim=-1)
    return image_embeds.cpu().numpy().astype(np.float32)

# ─────────────────────────────────────────
# 9. Metric Functions
# ─────────────────────────────────────────
def recall_at_k(retrieved_ids, relevant_ids, k):
    return 1.0 if len(set(retrieved_ids[:k]) & relevant_ids) > 0 else 0.0

def ndcg_at_k(retrieved_ids, relevant_ids, k, num_relevant):
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(num_relevant, k)))
    if idcg == 0: return 0.0
    dcg = sum(1.0 / np.log2(i + 2) for i, rid in enumerate(retrieved_ids[:k]) if rid in relevant_ids)
    return dcg / idcg

def average_precision_at_k(retrieved_ids, relevant_ids, k, num_relevant):
    hits, precision = 0, 0.0
    for i, rid in enumerate(retrieved_ids[:k]):
        if rid in relevant_ids:
            hits += 1
            precision += hits / (i + 1)
    if hits == 0: return 0.0
    return precision / min(num_relevant, k)

def compute_metrics(all_retrieved, all_relevant, all_num_relevant, k_values):
    metrics = {}
    for k in k_values:
        recalls = [recall_at_k(ret, rel, k) for ret, rel in zip(all_retrieved, all_relevant)]
        ndcgs = [ndcg_at_k(ret, rel, k, nr) for ret, rel, nr in zip(all_retrieved, all_relevant, all_num_relevant)]
        maps = [average_precision_at_k(ret, rel, k, nr) for ret, rel, nr in zip(all_retrieved, all_relevant, all_num_relevant)]

        metrics[f'Recall@{k}'] = float(np.mean(recalls))
        metrics[f'NDCG@{k}']   = float(np.mean(ndcgs))
        metrics[f'mAP@{k}']    = float(np.mean(maps))
    return metrics

# ─────────────────────────────────────────
# 10. Core Evaluation Function (Class-Aware)
# ─────────────────────────────────────────
def evaluate_index(index_path, metadata_path, clip_model, processor):
    index = faiss.read_index(index_path)
    hnsw  = faiss.downcast_index(index.index)
    hnsw.hnsw.efSearch = 128

    with open(metadata_path, 'r') as f:
        metadata = json.load(f)

    id_to_item = {int(k): v['item_id'] for k, v in metadata.items()}

    # Dictionaries to hold data for overall + classes
    class_data = {
        'overall': {'ret': [], 'rel': [], 'num': []},
        'Upper':   {'ret': [], 'rel': [], 'num': []},
        'Lower':   {'ret': [], 'rel': [], 'num': []},
        'Full':    {'ret': [], 'rel': [], 'num': []}
    }
    
    skipped = 0

    for _, row in tqdm(df_query.iterrows(), total=len(df_query), desc="  Queries"):
        query_item_id = row['item_id']
        image_name    = row['image_name']
        cls_name      = row['class_name']

        relevant_ids = {query_item_id}
        num_relevant = item_id_to_gallery_count.get(query_item_id, 1)

        crop_path = resolve_crop_path(image_name)
        if not crop_path:
            skipped += 1
            continue

        try:
            cropped_img = Image.open(crop_path).convert("RGB")
        except Exception:
            skipped += 1
            continue

        query_embed = get_query_embedding(clip_model, processor, cropped_img, DEVICE)
        D, I = index.search(query_embed, k=TOP_K_SEARCH)
        candidate_faiss_ids = I[0].tolist()

        retrieved_ids = [id_to_item.get(int(fid)) for fid in candidate_faiss_ids if id_to_item.get(int(fid)) is not None]

        # Log for overall
        class_data['overall']['ret'].append(retrieved_ids)
        class_data['overall']['rel'].append(relevant_ids)
        class_data['overall']['num'].append(num_relevant)
        
        # Log for specific class
        if cls_name in class_data:
            class_data[cls_name]['ret'].append(retrieved_ids)
            class_data[cls_name]['rel'].append(relevant_ids)
            class_data[cls_name]['num'].append(num_relevant)

    if skipped > 0:
        print(f"  ⚠️  Skipped {skipped} queries")

    # Compute metrics for overall + each class
    final_metrics = {}
    for c_name, data in class_data.items():
        if len(data['ret']) > 0:
            final_metrics[c_name] = compute_metrics(data['ret'], data['rel'], data['num'], K_VALUES)

    return final_metrics

# ─────────────────────────────────────────
# 11. ABLATION A
# ─────────────────────────────────────────
print(f"\n{'='*55}")
print("ABLATION A — Vision-only CLIP (α=1.0, frozen)")
print(f"{'='*55}")

clip_model = CLIPModel.from_pretrained(CLIP_CACHE).to(DEVICE)
processor  = CLIPProcessor.from_pretrained(CLIP_CACHE)
clip_model.eval()

a_metrics_nested = evaluate_index(
    os.path.join(faiss_index_dir, "A_vision_only_alpha1.0_index.bin"),
    os.path.join(faiss_index_dir, "A_vision_only_alpha1.0_metadata.json"),
    clip_model, processor
)
del clip_model
torch.cuda.empty_cache()

# ─────────────────────────────────────────
# 12. ABLATION B
# ─────────────────────────────────────────
print(f"\n{'='*55}")
print(f"ABLATION B — Frozen CLIP + BLIP-2 captions (α={ALPHA_VALUES})")
print(f"{'='*55}")

b_metrics_nested  = {}
clip_model = CLIPModel.from_pretrained(CLIP_CACHE).to(DEVICE)
processor  = CLIPProcessor.from_pretrained(CLIP_CACHE)
clip_model.eval()

for alpha in ALPHA_VALUES:
    name = f"B_frozen_alpha{alpha}"
    print(f"\n  Evaluating B — α={alpha}...")
    b_metrics_nested[alpha] = evaluate_index(
        os.path.join(faiss_index_dir, f"{name}_index.bin"),
        os.path.join(faiss_index_dir, f"{name}_metadata.json"),
        clip_model, processor
    )
del clip_model
torch.cuda.empty_cache()

# ─────────────────────────────────────────
# 13. ABLATION C
# ─────────────────────────────────────────
print(f"\n{'='*55}")
print(f"ABLATION C — Fine-tuned CLIP + BLIP-2 captions")
print(f"{'='*55}")

# Now tracking metrics explicitly per seed AND per alpha
c_seed_metrics_nested = defaultdict(lambda: defaultdict(dict))

for seed in SEEDS:
    seed_model_path = os.path.join(clip_finetuned_base, f'seed_{seed}', 'best')
    if not os.path.exists(seed_model_path): continue

    print(f"\n  Loading fine-tuned CLIP — seed {seed}...")
    clip_model = CLIPModel.from_pretrained(seed_model_path).to(DEVICE)
    processor  = CLIPProcessor.from_pretrained(seed_model_path)
    clip_model.eval()

    for alpha in ALPHA_VALUES:
        name = f"C_finetuned_seed{seed}_alpha{alpha}"
        print(f"\n  Evaluating C — seed={seed}, α={alpha}...")
        metrics = evaluate_index(
            os.path.join(faiss_index_dir, f"{name}_index.bin"),
            os.path.join(faiss_index_dir, f"{name}_metadata.json"),
            clip_model, processor
        )
        c_seed_metrics_nested[alpha][seed] = metrics

    del clip_model
    torch.cuda.empty_cache()

# Mean ± std across seeds for nested dicts
c_mean_nested, c_std_nested = {}, {}
for alpha in ALPHA_VALUES:
    seeds_dict = c_seed_metrics_nested[alpha]
    if not seeds_dict: continue
    
    seed_list = list(seeds_dict.values())
    c_mean_nested[alpha], c_std_nested[alpha] = {}, {}
    
    for cls_name in ['overall', 'Upper', 'Lower', 'Full']:
        if cls_name not in seed_list[0]: continue
        c_mean_nested[alpha][cls_name] = {}
        c_std_nested[alpha][cls_name]  = {}
        for metric_name in seed_list[0][cls_name].keys():
            values = [m[cls_name][metric_name] for m in seed_list]
            c_mean_nested[alpha][cls_name][metric_name] = float(np.mean(values))
            c_std_nested[alpha][cls_name][metric_name]  = float(np.std(values))

# ─────────────────────────────────────────
# 14. Final Results Table
# ─────────────────────────────────────────
print(f"\n{'='*85}")
print("ABLATION STUDY RESULTS (OVERALL + CLASS-WISE)")
print(f"{'='*85}")

header = f"{'Configuration':<46}"
for k in K_VALUES:
    header += f"  R@{k}   N@{k}   M@{k}"
print(header)
print("-" * 85)

def print_row(name, metrics_dict, std_dict=None):
    # Print overall first
    if 'overall' in metrics_dict:
        m = metrics_dict['overall']
        row = f"{name:<46}"
        for k in K_VALUES:
            row += f"  {m.get(f'Recall@{k}',0):.3f}  {m.get(f'NDCG@{k}',0):.3f}  {m.get(f'mAP@{k}',0):.3f}"
        print(row)
        if std_dict and 'overall' in std_dict:
            s = std_dict['overall']
            row2 = f"{'     ± std':<46}"
            for k in K_VALUES:
                row2 += f"  {s.get(f'Recall@{k}',0):.3f}  {s.get(f'NDCG@{k}',0):.3f}  {s.get(f'mAP@{k}',0):.3f}"
            print(row2)
            
    # Print classes indented
    for cls_name in ['Upper', 'Lower', 'Full']:
        if cls_name not in metrics_dict: continue
        prefix = f"  └─ {cls_name}"
        m = metrics_dict[cls_name]
        row = f"{prefix:<46}"
        for k in K_VALUES:
            row += f"  {m.get(f'Recall@{k}',0):.3f}  {m.get(f'NDCG@{k}',0):.3f}  {m.get(f'mAP@{k}',0):.3f}"
        print(row)
        if std_dict and cls_name in std_dict:
            s = std_dict[cls_name]
            row2 = f"{'       ± std':<46}"
            for k in K_VALUES:
                row2 += f"  {s.get(f'Recall@{k}',0):.3f}  {s.get(f'NDCG@{k}',0):.3f}  {s.get(f'mAP@{k}',0):.3f}"
            print(row2)

print_row("A: Vision-only (α=1.0)", a_metrics_nested)

for alpha in ALPHA_VALUES:
    print_row(f"B: Frozen CLIP + captions (α={alpha})", b_metrics_nested.get(alpha, {}))

# Print each individual seed for Ablation C
for alpha in ALPHA_VALUES:
    for seed, metrics_dict in c_seed_metrics_nested[alpha].items():
         print_row(f"C: Fine-tuned (Seed {seed}) (α={alpha})", metrics_dict)

# Print the Mean and Std for Ablation C
for alpha in ALPHA_VALUES:
    print("-" * 85)
    print_row(f"C: Fine-tuned MEAN ACROSS SEEDS (α={alpha})", c_mean_nested.get(alpha, {}), c_std_nested.get(alpha, {}))

print("-" * 85)

# ─────────────────────────────────────────
# 15. Save Results
# ─────────────────────────────────────────
# Save full nested JSON
all_results = {
    "A_vision_only"    : a_metrics_nested,
    "B_frozen"         : {str(a): m for a, m in b_metrics_nested.items()},
    "C_finetuned_mean" : {str(a): m for a, m in c_mean_nested.items()},
    "C_finetuned_std"  : {str(a): m for a, m in c_std_nested.items()},
    "C_finetuned_raw"  : {str(a): {str(s): m for s, m in sd.items()} for a, sd in c_seed_metrics_nested.items()}
}
json_path = os.path.join(results_dir, 'ablation_results.json')
with open(json_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\n✅ Results JSON : {json_path}")

# Flatten for CSV
rows = []
for cls_name, m in a_metrics_nested.items():
    rows.append({"config": f"A_vision_only_{cls_name}", **m})

for alpha in ALPHA_VALUES:
    if alpha in b_metrics_nested:
        for cls_name, m in b_metrics_nested[alpha].items():
            rows.append({"config": f"B_frozen_alpha{alpha}_{cls_name}", **m})

# Add raw seeds to CSV
for alpha in ALPHA_VALUES:
     for seed, metrics_dict in c_seed_metrics_nested[alpha].items():
         for cls_name, m in metrics_dict.items():
              rows.append({"config": f"C_finetuned_seed{seed}_alpha{alpha}_{cls_name}", **m})

for alpha in ALPHA_VALUES:
    if alpha in c_mean_nested:
        for cls_name, m in c_mean_nested[alpha].items():
            rows.append({"config": f"C_finetuned_alpha{alpha}_mean_{cls_name}", **m})
        for cls_name, s in c_std_nested[alpha].items():
            rows.append({"config": f"C_finetuned_alpha{alpha}_std_{cls_name}", **s})

csv_path = os.path.join(results_dir, 'ablation_results.csv')
pd.DataFrame(rows).to_csv(csv_path, index=False)
print(f"✅ Results CSV  : {csv_path}")

zip_path = shutil.make_archive(
    '/kaggle/working/eval_results',
    'zip',
    '/kaggle/working',
    'eval_results'
)
print(f"✅ Results ZIP  : {zip_path}")
print("\nDone! Use these numbers in your report.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ HuggingFace login successful.
GPUs available: 2
  GPU 0: Tesla T4 — 15.6 GB
  GPU 1: Tesla T4 — 15.6 GB


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...


  Evaluating C — seed=623, α=0.5...


  Queries: 100%|██████████| 14218/14218 [04:05<00:00, 57.94it/s]



  Evaluating C — seed=623, α=0.7...


  Queries: 100%|██████████| 14218/14218 [04:09<00:00, 57.03it/s]



ABLATION STUDY RESULTS (OVERALL + CLASS-WISE)
Configuration                                   R@5   N@5   M@5  R@10   N@10   M@10  R@15   N@15   M@15
-------------------------------------------------------------------------------------
A: Vision-only (α=1.0)                          0.386  0.173  0.125  0.453  0.176  0.119  0.491  0.182  0.119
  └─ Upper                                      0.357  0.151  0.106  0.425  0.152  0.099  0.463  0.157  0.099
  └─ Lower                                      0.443  0.206  0.151  0.505  0.205  0.140  0.548  0.211  0.140
  └─ Full                                       0.422  0.212  0.163  0.492  0.228  0.167  0.527  0.237  0.170
B: Frozen CLIP + captions (α=0.5)               0.381  0.171  0.124  0.457  0.179  0.121  0.503  0.186  0.122
  └─ Upper                                      0.357  0.154  0.109  0.433  0.160  0.105  0.480  0.166  0.106
  └─ Lower                                      0.415  0.188  0.137  0.491  0.193  0.130  0.538  0.201 